In [3]:
import hopsworks
from hsfs.feature_store import FeatureStore
from hsfs.feature_group import FeatureGroup

c:\Users\as296\anaconda3\envs\aqi_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
project = hopsworks.login(
    project='your_project_name',  # Replace with your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value="your api key"  # Get from Hopsworks UI > Account Settings > API Keys
)

In [11]:
api_key = os.environ.get("HOPSWORKS_API_KEY")
if not api_key:
    raise EnvironmentError(
            "HOPSWORKS_API_KEY is not set. Copy .env.example to .env, "
            "paste your key (Hopsworks -> your profile icon -> Settings -> "
            "API Keys -> New API Key), and make sure load_dotenv() has run "
            "before calling connect().")
project = hopsworks.login(
        api_key_value=api_key,
        project=os.environ.get("HOPSWORKS_PROJECT"),  # optional; None = your default project
    )


2026-08-21 21:20:29,276 INFO: Closing external client and cleaning up certificates.
2026-08-21 21:20:29,276 INFO: Connection closed.
2026-08-21 21:20:29,281 INFO: Initializing external client
2026-08-21 21:20:29,283 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/tmp\\eu-west.cloud.hopsworks.ai'

In [12]:
project

NameError: name 'project' is not defined

In [6]:
def connect() -> FeatureStore:
    """
    Connects to Hopsworks using HOPSWORKS_API_KEY (and optionally
    HOPSWORKS_PROJECT) from the environment, and returns the project's
    Feature Store handle.
    """

    api_key = os.environ.get("HOPSWORKS_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "HOPSWORKS_API_KEY is not set. Copy .env.example to .env, "
            "paste your key (Hopsworks -> your profile icon -> Settings -> "
            "API Keys -> New API Key), and make sure load_dotenv() has run "
            "before calling connect()."
        )
    project = hopsworks.login(
        api_key_value=api_key,
        project=os.environ.get("HOPSWORKS_PROJECT"),  # optional; None = your default project
    )
    
    return project.get_feature_store()

In [7]:
fs = connect()

2026-08-21 21:10:19,073 INFO: Initializing external client
2026-08-21 21:10:19,074 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/tmp\\eu-west.cloud.hopsworks.ai'

In [1]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()

open_weather_api_key = os.getenv("OPENWEATHER_KEY")


def geocode_city(city_name, api_key):
    url = "http://api.openweathermap.org/geo/1.0/direct"
    params = {"q": city_name, "limit": 1, "appid": api_key}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"No location found for '{city_name}'. Try a different format, e.g. 'City,CountryCode'.")
    return data[0]["lat"], data[0]["lon"], data[0]["name"], data[0]["country"]

if __name__ == "__main__":
    CITY_NAME = "Islamabad,PK" 
    lat, lon, resolved_name, country = geocode_city(CITY_NAME, open_weather_api_key)
    print(f"Resolved: {resolved_name}, {country}  ->  lat={lat}, lon={lon}")


Resolved: Islamabad, PK  ->  lat=33.6938118, lon=73.0651511


In [2]:
from datetime import datetime, timedelta, timezone

In [3]:
end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=80)

In [4]:
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

In [5]:
resp = requests.get(
        "http://api.openweathermap.org/data/2.5/air_pollution/history",
        params={
            "lat": 31.5656822,
            "lon": 74.3141829,
            "start": start_ts,
            "end": end_ts,
            "appid": open_weather_api_key
        },
        timeout=90,
    )
resp.raise_for_status()
data = resp.json()

In [6]:
import pandas as pd

In [7]:
records = []
for item in data.get("list", []):
        records.append({
            "timestamp": pd.to_datetime(item["dt"], unit="s", utc=True),
            "pm25": item["components"]["pm2_5"],
            "pm10": item["components"]["pm10"],
            "co": item["components"]["co"],
            "no2": item["components"]["no2"],
            "so2": item["components"]["so2"],
            "o3": item["components"]["o3"],
            "aqi": item["main"]["aqi"],  # Note: Scale is 1-5
        })

df = pd.DataFrame(records)

In [8]:
df

,timestamp,pm25,pm10,co,no2,so2,o3,aqi
0,2026-06-05 19:00:00+00:00,57.92,98.56,697.25,16.97,3.96,59.69,4
1,2026-06-05 20:00:00+00:00,62.90,101.35,754.50,16.75,3.41,52.98,4
2,2026-06-05 21:00:00+00:00,69.22,106.30,812.94,16.32,3.05,47.83,4
3,2026-06-05 22:00:00+00:00,76.77,112.69,870.38,15.89,2.76,43.25,5
4,2026-06-05 23:00:00+00:00,84.57,119.44,931.54,15.69,2.50,39.12,5
...,...,...,...,...,...,...,...,...
1867,2026-08-24 14:00:00+00:00,57.57,127.90,395.73,12.10,2.52,61.47,4
1868,2026-08-24 15:00:00+00:00,61.29,130.43,491.49,15.77,2.80,48.37,4
1869,2026-08-24 16:00:00+00:00,64.19,131.94,569.61,18.84,3.05,38.69,4
1870,2026-08-24 17:00:00+00:00,67.62,134.71,648.86,21.73,3.27,29.78,4


In [9]:
def fetch_historical_weather(lat: float, lon: float, start_ts: int, end_ts: int, api_key: str) -> pd.DataFrame:
    """
    Fetches historical weather data using the One Call API 3.0.
    Note: Requires subscribing to One Call 3.0 (Free up to 1,000 calls/day).
    """
    # Convert timestamps back to dates to iterate day-by-day
    start_dt = datetime.fromtimestamp(start_ts, tz=timezone.utc)
    end_dt = datetime.fromtimestamp(end_ts, tz=timezone.utc)
    
    records = []
    
    # One Call API 3.0 historical data requires fetching day-by-day
    current_dt = start_dt
    while current_dt <= end_dt:
        dt_unix = int(current_dt.timestamp())
        
        resp = requests.get(
            "https://api.openweathermap.org/data/3.0/onecall/timemachine",
            params={
                "lat": lat,
                "lon": lon,
                "dt": dt_unix,
                "units": "metric",
                "appid": api_key
            },
            timeout=100,
        )
        resp.raise_for_status()
        data = resp.json()
        
        for item in data.get("data", []):
            records.append({
                "timestamp": pd.to_datetime(item["dt"], unit="s", utc=True),
                "temp": item["temp"],
                "humidity": item["humidity"],
                "pressure": item["pressure"],
                "wind_speed": item["wind_speed"],
            })
            
        current_dt += timedelta(days=1)

    return pd.DataFrame(records)


In [10]:
weather_df = fetch_historical_weather(lat, lon, start_ts, end_ts, open_weather_api_key)

In [11]:
weather_df

,timestamp,temp,humidity,pressure,wind_speed
0,2026-06-05 18:14:36+00:00,27.01,64,1004,1.83
1,2026-06-06 18:14:36+00:00,27.46,41,1003,0.89
2,2026-06-07 18:14:36+00:00,29.25,40,1002,0.45
3,2026-06-08 18:14:36+00:00,28.90,47,999,0.45
4,2026-06-09 18:14:36+00:00,29.01,49,998,0.45
...,...,...,...,...,...
76,2026-08-20 18:14:36+00:00,25.92,92,1006,2.95
77,2026-08-21 18:14:36+00:00,28.14,89,1005,0.45
78,2026-08-22 18:14:36+00:00,28.14,87,1005,0.45
79,2026-08-23 18:14:36+00:00,27.59,76,1007,0.45
